# Module 29 — KV-Caching

Every generation loop so far (Modules 17, 18, 28) fed the model the
**entire growing sequence** at every single step — to generate token 200,
it recomputed attention over all 199 previous tokens from scratch, even
though their Key and Value vectors (Module 10) never change once computed.
Generating `n` tokens this way costs `1 + 2 + ... + n` = `O(n^2)` total
attention work.

**KV-caching** fixes this: store every layer's Key/Value vectors as
they're computed, and at each new step only compute Q/K/V for the *single
new token*, appending its K/V onto the cache. Total work drops to `O(n)`.
Critically, this must be a pure speed optimization — the actual generated
tokens must come out **identical** to the no-cache version. This module
proves that, then measures the speedup.

## 1. Attention with a query/key position offset

The core piece that makes caching work: when only the new token's Query is
being computed, but it needs to attend against *all* cached Keys/Values,
the causal mask has to account for the new query's real position in the
full sequence (`query_offset` positions already came before it).

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


def attention_with_offset(Q, K, V, query_offset):
    """Q may cover fewer positions than K/V - query i is really at
    absolute position (query_offset + i), and can attend to every key at
    or before that absolute position."""
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    seq_q, seq_k = scores.shape[-2], scores.shape[-1]
    q_positions = torch.arange(seq_q, device=Q.device) + query_offset
    k_positions = torch.arange(seq_k, device=Q.device)
    mask = k_positions.unsqueeze(0) > q_positions.unsqueeze(1)
    scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(*batch_dims, seq_len, num_heads, d_k).transpose(-3, -2)


def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    return t.transpose(-3, -2).contiguous().view(*batch_dims, seq_len, num_heads * d_k)


print("Offset-aware attention defined.")

## 2. A cache-aware transformer block and model

In [ ]:
class CachedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache=None):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        query_offset = 0
        if cache is not None:
            query_offset = cache["K"].shape[0]
            K = torch.cat([cache["K"], K], dim=0)
            V = torch.cat([cache["V"], V], dim=0)
        new_cache = {"K": K, "V": V}
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = attention_with_offset(Qh, Kh, Vh, query_offset)
        return self.Wo(merge_heads(out)), new_cache


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class CachedBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CachedMultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model)

    def forward(self, x, cache=None):
        attn_out, new_cache = self.attn(self.ln1(x), cache=cache)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache


class CachedGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([CachedBlock(d_model, num_heads) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight

    def forward(self, idx, caches=None, position_offset=0):
        seq_len = idx.shape[-1]
        positions = torch.arange(seq_len, device=idx.device) + position_offset
        x = self.token_embed(idx) + self.pos_embed(positions)
        new_caches = []
        for i, block in enumerate(self.blocks):
            x, new_cache = block(x, cache=(caches[i] if caches is not None else None))
            new_caches.append(new_cache)
        x = self.ln_f(x)
        return self.head(x), new_caches


torch.manual_seed(42)
vocab_size, d_model, num_heads, num_layers, max_seq_len = 100, 128, 8, 6, 300
model = CachedGPT(vocab_size, d_model, num_heads, num_layers, max_seq_len)
model.eval()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 3. Proving the outputs are identical, token for token

Greedy decoding (Module 28) to keep both paths deterministic. One
generation loop recomputes the full prefix every step (the Module
17/18/28 approach); the other feeds only the newest token, relying
entirely on the KV cache for everything before it.

In [ ]:
@torch.no_grad()
def generate_no_cache(model, prompt, n_new):
    idx = prompt.clone()
    for _ in range(n_new):
        logits, _ = model(idx)
        next_id = torch.argmax(logits[-1]).item()
        idx = torch.cat([idx, torch.tensor([next_id])])
    return idx


@torch.no_grad()
def generate_with_cache(model, prompt, n_new):
    idx_list = prompt.tolist()
    logits, caches = model(prompt, caches=None, position_offset=0)
    next_id = torch.argmax(logits[-1]).item()
    idx_list.append(next_id)
    for _ in range(n_new - 1):
        logits, caches = model(torch.tensor([next_id]), caches=caches, position_offset=len(idx_list) - 1)
        next_id = torch.argmax(logits[-1]).item()
        idx_list.append(next_id)
    return torch.tensor(idx_list)


prompt = torch.randint(0, vocab_size, (10,))
out_no_cache = generate_no_cache(model, prompt, 40)
out_cache = generate_with_cache(model, prompt, 40)

assert torch.equal(out_no_cache, out_cache)
print("no-cache: ", out_no_cache.tolist())
print("cached:   ", out_cache.tolist())
print("\nConfirmed: identical generated tokens, with or without caching.")

## 4. Measuring the actual speedup

In [ ]:
import time

n_new = 200
prompt = torch.randint(0, vocab_size, (10,))

start = time.time()
generate_no_cache(model, prompt, n_new)
no_cache_time = time.time() - start

start = time.time()
generate_with_cache(model, prompt, n_new)
cache_time = time.time() - start

print(f"generating {n_new} tokens, no cache:   {no_cache_time:.3f}s")
print(f"generating {n_new} tokens, with cache: {cache_time:.3f}s")
print(f"speedup: {no_cache_time / cache_time:.2f}x")
assert cache_time < no_cache_time

## Recap

- KV-caching was verified to be a pure speed optimization: greedy
  generation produces byte-for-byte identical tokens whether or not the
  cache is used.
- Measured directly: generating 200 tokens with caching was meaningfully
  faster than recomputing the full prefix at every step — and the gap
  grows with sequence length, since no-cache generation is `O(n^2)` while
  cached generation is `O(n)`.

Phase 5's final pieces — Module 30 sources the real training corpus, and
Module 31 runs the actual pretrain — are what everything from Modules 01
through 29 has been building toward.